In [36]:
import csv
import numpy as np

def import_herbier(herbier):
    L = []

    with open(herbier, newline='') as csvfile:
        spamreader = csv.reader(csvfile, delimiter = ',', quotechar='|')
        for row in spamreader:
            L.append(row.copy())

    return np.array(L)

print(L)



[['nom', 'nom_scientifique', 'famille', 'cycle', 'besoins', 'photo'], ['tomate', 'Solanum lycopersicum', 'Solanaceae', 'Annuelle', 'Plein soleil', 'https://example.com/tomate.jpg'], ['mais', 'Zea mays', 'Poaceae', 'Annuelle', 'Plein soleil', 'https://example.com/mais.jpg'], ['pomme de terre', 'Solanum tuberosum', 'Solanaceae', 'Vivace', 'Mi-ombre', 'https://example.com/pomme_de_terre.jpg']]


### 3. Cahier des charges
L'application devra gérer un ensemble de plantes. 
Chaque plante possède au minimum les informations suivantes : 
- nom vernaculaire ; 
-  nom scientifique ; 
-  famille botanique ; 
-  cycle ; 
-  besoins ; 
-  photo. 

### 4. Classe plante
méthode `__init__` et `__str__`

In [21]:
class plante:
    def __init__(self, nom, nom_scientifique, famille, cycle, besoins, photo):
        self.nom = nom
        self.nom_scientifique = nom_scientifique
        self.famille = famille
        self.cycle = cycle
        self.besoins = besoins
        self.photo = photo

    def __str__(self):
        return f"{self.nom} ({self.nom_scientifique}) - {self.famille} \n Cycle : {self.cycle} \n Besoins : {self.besoins}"


In [22]:
A = plante(1,2,3,4,5,6)
print(A)

1 (2) - 3 
 Cycle : 4 
 Besoins : 5


## 5. Gestion de l'herbier
contenant les méthodes : 
- `ajouter_plante(plante)`
- `supprimer_plante(...)`
- `rechercher(...)`
- `filter_par_famille(...)`
- `filtrer_par_cycle()`
- `filtrer_par_besoin(...)`
- `afficher()`

In [324]:
class herbier:
    def __init__(self):
        '''
        Class contenant les plantes et des méthodes d'actions sur ses plantes / l'herbier
        '''
        info_plantes = import_herbier('herbier.csv')
        n = len(info_plantes)
        self.carac = info_plantes[0,:]
        self.list_plante = info_plantes[1,:]
        for k in range(2,n):
            self.ajouter_plante(info_plantes[k,:])


    def ajouter_plante(self, info_plante): # list contenant les 6 carac
        if len(info_plante) == 6:
            self.list_plante = np.vstack((self.list_plante, info_plante))
        else:
            raise TypeError(f"la liste {info_plante} n'est pas  au bon format [nom, nom_scientifique, famille, cycle, besoins, photo]")


    def supprimer_plante(self, att):
        ''' 
        supprimer toutes les lignes qui contiennent au moins les attributs att dans la ligne
        '''
        Table = np.zeros(self.list_plante.shape)
        for k in range(len(att)):
            Table += np.where(self.list_plante == att[k],1,0)
        Sum = np.sum(Table, axis=1)
        max = np.where(Sum == np.max(Sum))[0]
        if int(Sum[max[0]]) == len(att):
            for k in range(len(max)):
                self.list_plante = np.delete(self.list_plante,max[k],0)

    def rechercher(self, rech, diff = 0):
        ''' 
        On va rechercher dans toutes nos plantes (defini chaucne par 6 attributs/caractère)
            lesquels correspondent à la description.
        Si un seul caractère (à diff pret) ne correspond pas avec notre description, la plante n'est pas affichée.
        '''
        list_recherche = []
        Table = np.zeros(self.list_plante.shape)

        rech = rech.replace("'", " ").split()
        n = 0

        for unit in rech:
            valeur = str(unit).strip().lower()
            valeurs_plantes = np.char.lower(np.char.strip(self.list_plante))

            t_Table = valeurs_plantes == valeur
            besoins = np.char.split(valeurs_plantes[:, 4], sep=';')
            t_Table[:, 4] = np.array([valeur in besoins_plante for besoins_plante in besoins])
            t_Table = np.where(t_Table, 1, 0)
            if t_Table.any() == 1:
                Table += t_Table
                print(Table)
                n += 1

        Sum = np.sum(Table, axis=1)
        max = np.where(Sum >= np.max(Sum)-diff)[0]
        if int(Sum[max[0]]) >= n - diff :
            for k in range(len(max)):
                list_recherche.append(self.list_plante[max[k]])

        if list_recherche == []:
            return 'aucune correspondance avec votre recherche'
        else:
            return [
                f"{plante[0]} ({plante[1]}) - {plante[2]}; "
                f"Cycle : {plante[3]}; "
                f"Besoins : {plante[4]}"
                for plante in list_recherche
            ]


    def nettoyer(self, ensemble = None):
        correspondance_encodage = {
    'Ã€': 'À',
    'Ã‚': 'Â',
    'Ã‡': 'Ç',
    'Ã‰': 'É',
    'Ãˆ': 'È',
    'ÃŠ': 'Ê',
    'Ã‹': 'Ë',
    'ÃŽ': 'Î',
    'Ã': 'à',
    'Ã¢': 'â',
    'Ã§': 'ç',
    'Ã©': 'é',
    'Ã¨': 'è',
    'Ãª': 'ê',
    'Ã«': 'ë',
    'Ã®': 'î',
    'Ã¯': 'ï',
    'Ã´': 'ô',
    'Ã¶': 'ö',
    'Ã¹': 'ù',
    'Ã»': 'û',
    'Ã¼': 'ü',
    'Ã¿': 'ÿ',
    'â‚¬': '€',
    'â€™': '’',
    'â€œ': '“',
    'â€': '”',
    'â€“': '–',
    'â€”': '—',
    'â€¦': '…',
}
        def traduire_encodage(texte):
            """Corrige les caractères UTF-8 affichés avec un mauvais encodage."""
            texte = str(texte)
            for erreur in sorted(correspondance_encodage, key=len, reverse=True):
                texte = texte.replace(erreur, correspondance_encodage[erreur])
            return texte
        
        if not ensemble:
            ensemble = self.list_plante

        ENSEMBLE = []
        e_temp = []
        for ligne in ensemble:
            print(ligne)
            for idx, mot in enumerate(ligne):
                print(mot, traduire_encodage(mot))
                traduire_encodage(mot)

                if idx != 5:
                    mot = mot.strip().lower() #enlever les espaces

                    if idx == 1 or idx == 2 or idx == 3:
                        mot = mot[0].upper() + mot[1:]
                e_temp.append(mot[idx])
            ENSEMBLE.append(e_temp)

        self.list_plante = ENSEMBLE

            





In [302]:
mot = 'tomate;2  '
mot = mot[0].upper() + mot[1:]
print(mot.strip())

Tomate;2


AttributeError: 'list' object has no attribute 'strip'

In [325]:
h = herbier()
h.nettoyer()
L = h.list_plante



['Tomate' 'Solanum lycopersicum' 'Solanaceae' 'annuel'
 'soleil;arrosage regulier' 'tomate.jpg']
Tomate Tomate
Solanum lycopersicum Solanum lycopersicum
Solanaceae Solanaceae
annuel annuel
soleil;arrosage regulier soleil;arrosage regulier
tomate.jpg tomate.jpg
['tomate ' 'Solanum lycopersicum' 'SolanacÃ©es' 'Annuel'
 'soleil;arrosage rÃ©gulier' 'tomate.jpg']
tomate  tomate 
Solanum lycopersicum Solanum lycopersicum
SolanacÃ©es Solanacées
Annuel Annuel
soleil;arrosage rÃ©gulier soleil;arrosage régulier
tomate.jpg tomate.jpg
['Tournesol' 'Helianthus annuus' 'Asteraceae' 'annuel' 'soleil'
 'tournesol.jpg']
Tournesol Tournesol
Helianthus annuus Helianthus annuus
Asteraceae Asteraceae
annuel annuel
soleil soleil
tournesol.jpg tournesol.jpg
['Haricot' 'Phaseolus vulgaris' 'Fabaceae' 'annuel'
 'soleil;arrosage moyen' 'haricot.jpg']
Haricot Haricot
Phaseolus vulgaris Phaseolus vulgaris
Fabaceae Fabaceae
annuel annuel
soleil;arrosage moyen soleil;arrosage moyen
haricot.jpg haricot.jpg
['Pois' '

IndexError: string index out of range

In [ ]:
print(h.list_plante)
h.supprimer_plante(['tomate','Annuelle'])
print(h.list_plante)

In [307]:
h.rechercher("tomate soleil", diff=0) #Pour rechercher toutes les lignes qui contiennent Solanaceae

#h.rechercher() #Pour tout rechercher = afficher


[[1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
[[1. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0.]]


['Tomate (Solanum lycopersicum) - Solanaceae; Cycle : annuel; Besoins : soleil;arrosage regulier',
 'tomate  (Solanum lycopersicum) - SolanacÃ©es; Cycle : Annuel; Besoins : soleil;arrosage rÃ©gulier']